# Prototype Hooks

This notebook validates that the residual steering hooks and head patching instrumentation work end-to-end on a toy example (TinyLlama).


In [ ]:
from src.models.llama_loader import load_model_and_tokenizer
from src.hooks.steering_hooks import capture_residual_stream, inject_steering_vector
from src.hooks.head_patch_hooks import ensure_head_patching, cache_mode, patch_mode
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model, tokenizer = load_model_and_tokenizer(model_name, device="cpu")
model.eval()

prompt = "Which planet is known as the Red Planet?"
inputs = tokenizer(prompt, return_tensors="pt")
pos_idx = inputs["input_ids"].shape[1] - 1

# Residual capture example
with capture_residual_stream(model, layer_idx=0, pos_idx=pos_idx) as storage:
    _ = model(**inputs)
print("Captured residual keys:", storage.keys())

# Steering injection example (adds zeros to avoid changing output)
vector = torch.zeros_like(storage[0].tensor)
with inject_steering_vector(model, layer_idx=0, pos_idx=pos_idx, vector=vector, scale=0.0):
    _ = model(**inputs)
print("Steering injection hook executed")

# Head cache + patch example
registry = ensure_head_patching(model)
example_id = "demo"
with cache_mode(registry, example_id, pos_idx):
    _ = model(**inputs)
with patch_mode(registry, example_id, layer_idx=0, head_idx=0, pos_idx=pos_idx):
    _ = model(**inputs)
print("Head patch executed for layer 0, head 0")
